# Hybrid Quantum Model for Atmosphere Prediction

This notebook implements a hybrid auxiliary+spectral MLP-VQC-MLP model that:
- Loads ADC2023 auxiliary metadata and 52-bin spectra from the local dataset copy
- Encodes auxiliary and spectral inputs separately, then fuses them into a 5-D latent
- Feeds the fused latent into a 5-qubit VQC (estimator, adjoint gradients) -> 5 expectation values
- Post-processes with a 2-layer MLP and residual from the encoder latent -> atmosphere parameters

**Data:** Training from `data/ariel-ml-dataset/TrainingData/`, test from `data/ariel-ml-dataset/TestData/`.



## 1. Environment setup

In [ ]:
# Standard library
import pickle
import random
import time
from collections import defaultdict
from pathlib import Path

# Scientific computing
import h5py
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Machine learning
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Quantum (Qiskit)
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter, ParameterExpression, ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.gradients import ParamShiftEstimatorGradient

# Adjoint gradient support from qiskit-algorithms
try:
    from qiskit_algorithms.gradients import ReverseEstimatorGradient
    from qiskit_algorithms.gradients.utils import GradientCircuit
    import qiskit_algorithms.gradients.utils as qiskit_grad_utils
    import qiskit_algorithms.gradients.base.base_estimator_gradient as qiskit_base_estimator_gradient
    _adjoint_available = True
except ImportError:
    _adjoint_available = False


def enable_qiskit_adjoint_patch() -> None:
    """Patch qiskit-algorithms so adjoint gradients work with symbolic CRX/CRY params.

    In the current stack, some controlled rotation gates still expose symbolic parameters
    while reporting ``is_parameterized() == False``. That leaves original ``theta[i]``
    symbols inside the internal gradient circuit and causes ``KeyError`` during backward.
    We patch the helper used by ``BaseEstimatorGradient._preprocess`` so every symbolic
    gate parameter gets a unique gradient parameter, regardless of that boolean.
    """

    if not _adjoint_available:
        return

    def _patched_assign_unique_parameters(circuit: QuantumCircuit) -> GradientCircuit:
        gradient_circuit = circuit.copy_empty_like(f"{circuit.name}_gradient")
        parameter_map = defaultdict(list)
        gradient_parameter_map = {}
        num_gradient_parameters = 0

        for instruction in circuit.data:
            op = (
                instruction.operation.to_mutable()
                if hasattr(instruction.operation, "to_mutable")
                else instruction.operation.copy()
            )
            op_params = list(getattr(op, "params", []))
            has_symbolic_params = any(getattr(angle, "parameters", None) for angle in op_params)

            if has_symbolic_params:
                new_op_params = []
                for angle in op_params:
                    if getattr(angle, "parameters", None):
                        new_parameter = Parameter(f"__gtheta{num_gradient_parameters}")
                        new_op_params.append(new_parameter)
                        num_gradient_parameters += 1
                        for parameter in angle.parameters:
                            parameter_map[parameter].append((new_parameter, angle.gradient(parameter)))
                        gradient_parameter_map[new_parameter] = angle
                    else:
                        new_op_params.append(angle)
                op.params = new_op_params

            gradient_circuit.append(op, instruction.qubits, instruction.clbits)

        gradient_circuit.global_phase = circuit.global_phase
        if isinstance(gradient_circuit.global_phase, ParameterExpression):
            substitution_map = {}
            for parameter in gradient_circuit.global_phase.parameters:
                if parameter in parameter_map:
                    substitution_map[parameter] = parameter_map[parameter][0][0]
                else:
                    new_parameter = Parameter(f"__gtheta{num_gradient_parameters}")
                    substitution_map[parameter] = new_parameter
                    parameter_map[parameter].append((new_parameter, 1))
                    num_gradient_parameters += 1
            gradient_circuit.global_phase = gradient_circuit.global_phase.subs(substitution_map)

        return GradientCircuit(gradient_circuit, parameter_map, gradient_parameter_map)

    qiskit_grad_utils._assign_unique_parameters = _patched_assign_unique_parameters
    qiskit_base_estimator_gradient._assign_unique_parameters = _patched_assign_unique_parameters


USE_ADJOINT_GRADIENT = _adjoint_available
if USE_ADJOINT_GRADIENT:
    enable_qiskit_adjoint_patch()
    print("Using ReverseEstimatorGradient (adjoint) with Qiskit compatibility patch.")
else:
    print("ReverseEstimatorGradient unavailable; using ParamShiftEstimatorGradient for backward.")



In [3]:
def set_random_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

RANDOM_SEED = 42
set_random_seed(RANDOM_SEED)

## 2. Data loading and preprocessing

Load auxiliary tables, FM targets, and 52-bin spectra keyed by `planet_ID`, then standardize each modality and build train/test tensors.



In [ ]:
# Paths relative to project root
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT.name == "models":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "ariel-ml-dataset"
AUX_TRAIN = DATA_ROOT / "TrainingData" / "AuxillaryTable.csv"
AUX_TEST = DATA_ROOT / "TestData" / "AuxillaryTable.csv"
SPEC_TRAIN = DATA_ROOT / "TrainingData" / "SpectralData.hdf5"
SPEC_TEST = DATA_ROOT / "TestData" / "SpectralData.hdf5"
FM_TRAIN = DATA_ROOT / "TrainingData" / "Ground Truth Package" / "FM_Parameter_Table.csv"

# Feature columns from auxiliary table (exclude planet_ID)
AUX_FEATURE_COLS = [
    "star_distance", "star_mass_kg", "star_radius_m", "star_temperature",
    "planet_mass_kg", "planet_orbital_period", "planet_distance", "planet_surface_gravity",
]

# Target columns from FM Parameter Table
TARGET_COLS = [
    "planet_radius", "planet_temp", "log_H2O", "log_CO2", "log_CO", "log_CH4", "log_NH3",
]

SPECTRAL_FIELDS = [
    "instrument_wlgrid",
    "instrument_spectrum",
    "instrument_noise",
    "instrument_width",
]


class SpectralStandardizer:
    def __init__(
        self,
        spectrum_mean: np.ndarray,
        spectrum_scale: np.ndarray,
        noise_mean: float,
        noise_scale: float,
        wl_mean: float,
        wl_scale: float,
        width_mean: float,
        width_scale: float,
    ) -> None:
        self.spectrum_mean = spectrum_mean.astype(np.float32)
        self.spectrum_scale = spectrum_scale.astype(np.float32)
        self.noise_mean = float(noise_mean)
        self.noise_scale = float(noise_scale)
        self.wl_mean = float(wl_mean)
        self.wl_scale = float(wl_scale)
        self.width_mean = float(width_mean)
        self.width_scale = float(width_scale)

    @classmethod
    def fit(cls, raw_spectra: np.ndarray) -> "SpectralStandardizer":
        raw64 = raw_spectra.astype(np.float64, copy=False)
        spectrum = raw64[:, :, 1]
        noise = raw64[:, :, 2]
        wl = raw64[0, :, 0]
        width = raw64[0, :, 3]

        spectrum_mean = spectrum.mean(axis=0)
        spectrum_scale = spectrum.std(axis=0)
        spectrum_scale = np.where(spectrum_scale == 0.0, 1.0, spectrum_scale)

        noise_mean = float(noise.mean())
        noise_scale = float(noise.std()) or 1.0
        wl_mean = float(wl.mean())
        wl_scale = float(wl.std()) or 1.0
        width_mean = float(width.mean())
        width_scale = float(width.std()) or 1.0

        return cls(
            spectrum_mean=spectrum_mean,
            spectrum_scale=spectrum_scale,
            noise_mean=noise_mean,
            noise_scale=noise_scale,
            wl_mean=wl_mean,
            wl_scale=wl_scale,
            width_mean=width_mean,
            width_scale=width_scale,
        )

    def transform(self, raw_spectra: np.ndarray) -> np.ndarray:
        spectrum = (raw_spectra[:, :, 1] - self.spectrum_mean) / self.spectrum_scale
        noise = (raw_spectra[:, :, 2] - self.noise_mean) / self.noise_scale
        wl = (raw_spectra[:, :, 0] - self.wl_mean) / self.wl_scale
        width = (raw_spectra[:, :, 3] - self.width_mean) / self.width_scale
        return np.stack([spectrum, noise, wl, width], axis=1).astype(np.float32)


def load_fm_targets(path: Path, columns: list[str]) -> pd.DataFrame:
    fm = pd.read_csv(path)
    if fm.columns[0] != "planet_ID" and "planet_ID" not in fm.columns:
        fm = fm.rename(columns={fm.columns[0]: "planet_ID"})
    return fm[["planet_ID", *columns]].copy()


def load_spectra(hdf5_path: Path, planet_ids: np.ndarray) -> np.ndarray:
    spectra = np.empty((len(planet_ids), 52, 4), dtype=np.float32)
    canonical_wl = None
    canonical_width = None

    with h5py.File(hdf5_path, "r") as handle:
        for idx, planet_id in enumerate(planet_ids):
            group = handle[f"Planet_{planet_id}"]
            wlgrid = group[SPECTRAL_FIELDS[0]][:].astype(np.float32)
            spectrum = group[SPECTRAL_FIELDS[1]][:].astype(np.float32)
            noise = group[SPECTRAL_FIELDS[2]][:].astype(np.float32)
            width = group[SPECTRAL_FIELDS[3]][:].astype(np.float32)

            if canonical_wl is None:
                canonical_wl = wlgrid
                canonical_width = width
            else:
                if not np.allclose(wlgrid, canonical_wl, atol=1.0e-8):
                    raise ValueError(f"{hdf5_path} has non-constant instrument_wlgrid.")
                if not np.allclose(width, canonical_width, atol=1.0e-8):
                    raise ValueError(f"{hdf5_path} has non-constant instrument_width.")

            spectra[idx, :, 0] = wlgrid
            spectra[idx, :, 1] = spectrum
            spectra[idx, :, 2] = noise
            spectra[idx, :, 3] = width

    return spectra



In [ ]:
# Load CSVs and align targets on planet_ID
aux_train_df = pd.read_csv(AUX_TRAIN)
aux_test_df = pd.read_csv(AUX_TEST)
fm_train_df = load_fm_targets(FM_TRAIN, TARGET_COLS)
train_df = aux_train_df.merge(fm_train_df, on="planet_ID", how="inner")

print(f"Project root: {PROJECT_ROOT}")
print(f"Training samples after join: {len(train_df)}")
print(f"Auxiliary feature columns: {AUX_FEATURE_COLS}")
print(f"Target columns: {TARGET_COLS}")

train_planet_ids = train_df["planet_ID"].to_numpy()
test_planet_ids = aux_test_df["planet_ID"].to_numpy()
train_raw_spectra = load_spectra(SPEC_TRAIN, train_planet_ids)
test_raw_spectra = load_spectra(SPEC_TEST, test_planet_ids)

print(f"Training spectra shape: {train_raw_spectra.shape}")
print(f"Test spectra shape: {test_raw_spectra.shape}")



In [ ]:
# Extract feature and target arrays for training
X_train_aux = train_df[AUX_FEATURE_COLS].values.astype(np.float32)
y_train = train_df[TARGET_COLS].values.astype(np.float32)
X_test_aux = aux_test_df[AUX_FEATURE_COLS].values.astype(np.float32)

# Scale auxiliary features and targets
scaler_X = StandardScaler()
X_train_aux_scaled = scaler_X.fit_transform(X_train_aux)
X_test_aux_scaled = scaler_X.transform(X_test_aux)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)

# Standardize spectra into 4 channels: spectrum, noise, wavelength grid, instrument width
spectral_scaler = SpectralStandardizer.fit(train_raw_spectra)
X_train_spectra_scaled = spectral_scaler.transform(train_raw_spectra)
X_test_spectra_scaled = spectral_scaler.transform(test_raw_spectra)

# Convert to PyTorch tensors
X_train_aux_tensor = torch.tensor(X_train_aux_scaled, dtype=torch.float32)
X_train_spectra_tensor = torch.tensor(X_train_spectra_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)
X_test_aux_tensor = torch.tensor(X_test_aux_scaled, dtype=torch.float32)
X_test_spectra_tensor = torch.tensor(X_test_spectra_scaled, dtype=torch.float32)

# CPU-first training config for the current StatevectorEstimator + TorchConnector path.
BATCH_SIZE = 128
train_dataset = TensorDataset(X_train_aux_tensor, X_train_spectra_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

n_train = X_train_aux_tensor.shape[0]
n_aux_features = X_train_aux_tensor.shape[1]
n_spectral_channels = X_train_spectra_tensor.shape[1]
n_spectral_bins = X_train_spectra_tensor.shape[2]
n_targets = len(TARGET_COLS)
print(f"Training size: {n_train}, aux features: {n_aux_features}, spectral tensor: {tuple(X_train_spectra_tensor.shape[1:])}")
print(f"Test samples: aux={X_test_aux_tensor.shape[0]}, spectra={X_test_spectra_tensor.shape[0]}")
print("Training path: CPU-first StatevectorEstimator + TorchConnector")



## 3. Dual encoder

Encode auxiliary metadata with an MLP and 52-bin spectra with a lightweight 1D CNN, then fuse both embeddings into a 5-D latent `z_enc` scaled to `[0, π]` for the quantum circuit.



In [ ]:
class AuxEncoder(nn.Module):
    """MLP encoder for auxiliary tabular inputs."""

    def __init__(self, in_dim: int, hidden_dim: int = 64, out_dim: int = 32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class SpectralEncoder(nn.Module):
    """Compact Conv1d encoder for 4-channel spectral inputs with 52 bins."""

    def __init__(self, in_channels: int = 4, hidden_dim: int = 64, out_dim: int = 32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Conv1d(32, hidden_dim, kernel_size=5, stride=2, padding=2),
            nn.GELU(),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_dim, out_dim),
            nn.GELU(),
            nn.Dropout(0.1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.proj(self.conv(x))


class FusionEncoder(nn.Module):
    """Fuse auxiliary and spectral embeddings into 5 latent angles."""

    def __init__(self, aux_dim: int = 32, spec_dim: int = 32, hidden_dim: int = 32, out_dim: int = 5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(aux_dim + spec_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, aux_feat: torch.Tensor, spectral_feat: torch.Tensor) -> torch.Tensor:
        fused = torch.cat([aux_feat, spectral_feat], dim=-1)
        return torch.sigmoid(self.net(fused)) * np.pi



## 4. 5-qubit VQC (estimator + adjoint gradient)

Reuse ansatz style from model_simulator: RY/RX rotations + CRX/CRY ring entanglement. Measure Pauli-Z on each qubit -> 5 expectation values.

In [8]:
def ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    """Hardware-efficient ansatz: RY/CRX and RX/CRY layers with ring topology."""
    theta = ParameterVector("θ", 2 * n_qubits * depth)
    qc = QuantumCircuit(n_qubits)
    param_idx = 0
    for _ in range(depth // 2):
        for i in range(n_qubits):
            qc.ry(theta[param_idx], i)
            param_idx += 1
        for i in range(n_qubits):
            qc.crx(theta[param_idx], i, (i + 1) % n_qubits)
            param_idx += 1
        for i in range(n_qubits):
            qc.rx(theta[param_idx], i)
            param_idx += 1
        for i in range(n_qubits):
            qc.cry(theta[param_idx], i, (i - 1) % n_qubits)
            param_idx += 1
    return qc


def create_angle_encoding(num_qubits: int) -> QuantumCircuit:
    """Angle encoding: RY(x_i) on qubit i for each input component."""
    qc = QuantumCircuit(num_qubits)
    params = ParameterVector("x", num_qubits)
    for i in range(num_qubits):
        qc.ry(params[i], i)
    return qc

In [9]:
# Keep the original circuit for backward so adjoint sees the true x/theta ordering.
class _AdjointEstimatorQNN(EstimatorQNN):
    def _backward(self, input_data, weights):
        parameter_values, num_samples = self._preprocess_forward(input_data, weights)
        input_grad, weights_grad = None, None
        if np.prod(parameter_values.shape) > 0:
            num_observables = self.output_shape[0]
            num_circuits = num_samples * num_observables
            circuits = [self._org_circuit] * num_circuits
            observables = [op for op in self._observables for _ in range(num_samples)]
            param_values = np.tile(parameter_values, (num_observables, 1))
            job = None
            if self._input_gradients:
                job = self.gradient.run(circuits, observables, param_values)
            elif len(parameter_values[0]) > self._num_inputs:
                params = [self._org_circuit.parameters[self._num_inputs :]] * num_circuits
                job = self.gradient.run(circuits, observables, param_values, parameters=params)
            if job is not None:
                from qiskit_machine_learning.exceptions import QiskitMachineLearningError
                try:
                    results = job.result()
                except Exception as exc:
                    raise QiskitMachineLearningError(f"Estimator job failed. {exc}") from exc
                input_grad, weights_grad = self._backward_postprocess(num_samples, results)
        return input_grad, weights_grad

NUM_QUBITS = 5
ANSATZ_DEPTH = 2  # must be even

ansatz_circuit = ansatz(NUM_QUBITS, ANSATZ_DEPTH)
feature_map = create_angle_encoding(NUM_QUBITS)

# Full circuit: feature_map -> ansatz
qc = QuantumCircuit(NUM_QUBITS)
qc.compose(feature_map, qubits=range(NUM_QUBITS), inplace=True)
qc.compose(ansatz_circuit, inplace=True)

input_params = list(feature_map.parameters)
weight_params = list(ansatz_circuit.parameters)

# 5 observables: Pauli-Z on qubit 0, 1, 2, 3, 4 (Pauli string: rightmost = qubit 0)
observables = [
    SparsePauliOp.from_list([("I" * (NUM_QUBITS - 1 - i) + "Z" + "I" * i, 1.0)])
    for i in range(NUM_QUBITS)
]

estimator = StatevectorEstimator()
if USE_ADJOINT_GRADIENT:
    gradient = ReverseEstimatorGradient()
else:
    gradient = ParamShiftEstimatorGradient(estimator)

QNNClass = _AdjointEstimatorQNN if USE_ADJOINT_GRADIENT else EstimatorQNN
qnn = QNNClass(
    circuit=qc,
    observables=observables,
    input_params=input_params,
    weight_params=weight_params,
    estimator=estimator,
    gradient=gradient,
)

In [10]:
class QuantumBlock(nn.Module):
    """Wraps EstimatorQNN (5 qubits -> 5 expectation values) via TorchConnector."""

    def __init__(self, qnn):
        super().__init__()
        self.quantum_layer = TorchConnector(qnn)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, 5) angles -> output (batch, 5) expectation values
        return self.quantum_layer(x)

## 5. Post-VQC 2-layer MLP head with residual from fused encoder output

Head input: concat of VQC output (5) and fused encoder latent (5) = 10. Residual: `out = head(head_in) + res_proj(z_enc)`.



In [11]:
class AtmosphereHead(nn.Module):
    """2-layer MLP: [q_feat, z_enc] (dim 10) -> hidden -> n_targets. Optional residual from z_enc."""

    def __init__(self, in_dim: int = 10, hidden_dim: int = 32, n_targets: int = 7, latent_dim: int = 5):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, n_targets),
        )
        self.res_proj = nn.Linear(latent_dim, n_targets)

    def forward(self, head_in: torch.Tensor, z_enc: torch.Tensor) -> torch.Tensor:
        out = self.mlp(head_in) + self.res_proj(z_enc)
        return out

## 6. End-to-end hybrid model



In [ ]:
class HybridAtmosphereModel(nn.Module):
    """Aux encoder + spectral encoder -> fused latent -> VQC -> residual head."""

    def __init__(self, aux_encoder, spectral_encoder, fusion_encoder, quantum_block, head):
        super().__init__()
        self.aux_encoder = aux_encoder
        self.spectral_encoder = spectral_encoder
        self.fusion_encoder = fusion_encoder
        self.quantum_block = quantum_block
        self.head = head

    def forward(self, x_aux: torch.Tensor, x_spectra: torch.Tensor) -> torch.Tensor:
        aux_feat = self.aux_encoder(x_aux)
        spectral_feat = self.spectral_encoder(x_spectra)
        z_enc = self.fusion_encoder(aux_feat, spectral_feat)
        q_feat = self.quantum_block(z_enc)
        head_in = torch.cat([q_feat, z_enc], dim=-1)
        return self.head(head_in, z_enc)



In [ ]:
# Build model components and full model
aux_encoder = AuxEncoder(in_dim=n_aux_features, hidden_dim=64, out_dim=32)
spectral_encoder = SpectralEncoder(in_channels=n_spectral_channels, hidden_dim=64, out_dim=32)
fusion_encoder = FusionEncoder(aux_dim=32, spec_dim=32, hidden_dim=32, out_dim=5)
quantum_block = QuantumBlock(qnn)
head = AtmosphereHead(in_dim=10, hidden_dim=32, n_targets=n_targets, latent_dim=5)
model = HybridAtmosphereModel(aux_encoder, spectral_encoder, fusion_encoder, quantum_block, head)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Total trainable parameters: {n_params}")



## 7. Training and evaluation

Use MSE on scaled targets. Validation split comes from the training set, while the public test split is kept for inference-only predictions. Track loss and per-target RMSE.



In [ ]:
# Validation split from training data (20% for validation)
VAL_MONITOR_SAMPLES = 2048

indices = np.arange(n_train)
idx_train, idx_val = train_test_split(indices, test_size=0.2, random_state=RANDOM_SEED)

X_aux_tr = X_train_aux_tensor[idx_train]
X_spec_tr = X_train_spectra_tensor[idx_train]
y_tr = y_train_tensor[idx_train]
X_aux_val = X_train_aux_tensor[idx_val]
X_spec_val = X_train_spectra_tensor[idx_val]
y_val = y_train_tensor[idx_val]

train_loader = DataLoader(
    TensorDataset(X_aux_tr, X_spec_tr, y_tr),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
val_dataset = TensorDataset(X_aux_val, X_spec_val, y_val)

if VAL_MONITOR_SAMPLES is not None and VAL_MONITOR_SAMPLES < len(y_val):
    val_subset_gen = torch.Generator().manual_seed(RANDOM_SEED)
    val_monitor_idx = torch.randperm(len(y_val), generator=val_subset_gen)[:VAL_MONITOR_SAMPLES]
    X_aux_val_monitor = X_aux_val[val_monitor_idx]
    X_spec_val_monitor = X_spec_val[val_monitor_idx]
    y_val_monitor = y_val[val_monitor_idx]
else:
    X_aux_val_monitor = X_aux_val
    X_spec_val_monitor = X_spec_val
    y_val_monitor = y_val

print(f"Train batches: {len(train_loader)}, validation samples: {len(y_val)}")
print(f"Validation monitor samples per epoch: {len(y_val_monitor)}")



In [ ]:
def train_epoch(model, loader, optimizer, loss_fn, epoch_idx=None, total_epochs=None):
    model.train()
    total_loss = 0.0
    n_batches = 0
    epoch_start = time.perf_counter()

    if epoch_idx is not None and total_epochs is not None:
        progress_desc = f"Epoch {epoch_idx + 1}/{total_epochs}"
    else:
        progress_desc = "Training"

    progress_bar = tqdm(loader, total=len(loader), desc=progress_desc, leave=False)
    for X_aux_b, X_spec_b, y_b in progress_bar:
        optimizer.zero_grad(set_to_none=True)
        pred = model(X_aux_b, X_spec_b)
        loss = loss_fn(pred, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
        progress_bar.set_postfix(batch=n_batches, avg_loss=f"{total_loss / n_batches:.4f}")

    epoch_seconds = time.perf_counter() - epoch_start
    return total_loss / max(n_batches, 1), epoch_seconds


def evaluate(model, X_aux, X_spectra, y, loss_fn, scaler_y=None):
    model.eval()
    eval_start = time.perf_counter()

    with torch.inference_mode():
        pred = model(X_aux, X_spectra)
        loss = loss_fn(pred, y).item()
        pred_np = pred.detach().cpu().numpy()
        y_np = y.detach().cpu().numpy()

    rmse_per_target = np.sqrt(np.mean((pred_np - y_np) ** 2, axis=0))
    if scaler_y is not None:
        pred_orig = scaler_y.inverse_transform(pred_np)
        y_orig = scaler_y.inverse_transform(y_np)
        rmse_orig = np.sqrt(np.mean((pred_orig - y_orig) ** 2, axis=0))
    else:
        rmse_orig = rmse_per_target

    eval_seconds = time.perf_counter() - eval_start
    return loss, rmse_per_target, rmse_orig, eval_seconds



In [16]:
EPOCHS = 20
LR = 1e-3
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_losses = []
val_losses = []
train_epoch_times = []
val_times = []

In [ ]:
print(
    f"Training will use fixed batch size {BATCH_SIZE} with {len(train_loader)} batches per epoch; "
    f"spectral input shape per sample = ({n_spectral_channels}, {n_spectral_bins})"
)



In [ ]:
for epoch in range(EPOCHS):
    tl, train_seconds = train_epoch(model, train_loader, optimizer, loss_fn, epoch_idx=epoch, total_epochs=EPOCHS)
    vl, rmse_scaled, rmse_orig, val_seconds = evaluate(
        model,
        X_aux_val_monitor,
        X_spec_val_monitor,
        y_val_monitor,
        loss_fn,
        scaler_y,
    )

    train_losses.append(tl)
    val_losses.append(vl)
    train_epoch_times.append(train_seconds)
    val_times.append(val_seconds)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | Train loss: {tl:.6f} | "
        f"Monitor val loss: {vl:.6f} | Monitor val RMSE (orig): {rmse_orig.mean():.4f} | "
        f"Train: {train_seconds:.2f}s | Val: {val_seconds:.2f}s"
    )

print(
    f"Average train epoch time: {np.mean(train_epoch_times):.2f}s | "
    f"Average monitor validation time: {np.mean(val_times):.2f}s"
)



In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train loss")
plt.plot(val_losses, label="Monitor validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.title("Training and monitor-validation loss")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(train_epoch_times, label="Train seconds per epoch")
plt.plot(val_times, label="Monitor validation seconds")
plt.xlabel("Epoch")
plt.ylabel("Seconds")
plt.legend()
plt.title("Training and validation timing")
plt.tight_layout()
plt.show()

In [ ]:
# Final full-validation metrics (per-target RMSE in original scale)
_, _, rmse_orig, final_val_seconds = evaluate(model, X_aux_val, X_spec_val, y_val, loss_fn, scaler_y)
print(f"Full validation time: {final_val_seconds:.2f}s")
print("Validation RMSE per target (original scale):")
for col, r in zip(TARGET_COLS, rmse_orig):
    print(f"  {col}: {r:.4f}")
print(f"Mean RMSE: {rmse_orig.mean():.4f}")



In [ ]:
# Save model and scalers for reuse
torch.save(model.state_dict(), "hybrid_atmosphere_model.pt")
with open("scaler_X.pkl", "wb") as f:
    pickle.dump(scaler_X, f)
with open("scaler_y.pkl", "wb") as f:
    pickle.dump(scaler_y, f)
with open("spectral_scaler.pkl", "wb") as f:
    pickle.dump(spectral_scaler, f)
print("Saved model state and all scalers.")



In [ ]:
# Inference on test set (no ground truth); save predictions if needed
model.eval()
with torch.no_grad():
    test_pred_scaled = model(X_test_aux_tensor, X_test_spectra_tensor)
test_pred_orig = scaler_y.inverse_transform(test_pred_scaled.cpu().numpy())
print(f"Test set predictions shape: {test_pred_orig.shape}")
# Optional: save to CSV with planet_IDs from aux_test_df
# test_out = pd.DataFrame(test_pred_orig, columns=TARGET_COLS)
# test_out.insert(0, "planet_ID", aux_test_df["planet_ID"].values)
# test_out.to_csv("test_predictions.csv", index=False)

